# Stage 0 — Project setup and assumptions

_Pipeline stage 0 of 14. This is the self-contained deep dive for the stage: narrative + analysis + interpretation, using the canonical `channel_heads` package and on-disk artifacts. Heavy rebuilds run via the `channel-heads` CLI (commands are given inline)._

## What this project does, and the contracts everything rests on

We detect **channel-head coupling** in drainage networks — pairs of channel heads
that first meet at a confluence whose contributing areas are spatially *touching* —
and quantify how common that coupling is on **Mars**, with a classifier **trained on
Earth** (after *Goren & Shelef 2024*).

Coupling is a fingerprint of **drainage-divide mobility**. When two growing channel
heads contest the same divide, their basins press together and touch. The *coupled
fraction* of confluences is therefore a proxy for how dynamic a landscape's divides
were — and Earth vs Mars asks whether martian valley networks froze in a fluvially
active or a degraded state.

This notebook fixes the **assumptions and frozen contracts** the whole pipeline
depends on, and verifies them against the code/artifacts on disk.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import channel_heads as ch
from channel_heads.io.paths import PROJECT_ROOT, RESULTS_DIR, EXAMPLE_DEMS
ROOT     = PROJECT_ROOT
MODELS   = ROOT / 'models'
MARS_OUT = ROOT / 'data/Mars/model_outputs'
MARS_IN  = ROOT / 'data/Mars/model_inputs'
MARS_DIR = ROOT / 'data/Mars'
REGIMES_ = ['regA', 'regB', 'regC']
RC = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}
MODEL_FEATURES = ['orientation_diff_deg','headhead_dist_norm','apex_angle_deg',
                  'strahler_order_diff','proximity_profile_norm']
OP_THR = {'regA': 0.756326, 'regB': 0.779264, 'regC': 0.759369}
def _abs(p):
    p = Path(p); return p if p.is_absolute() else ROOT / p
print('channel_heads', ch.__version__, '| root', ROOT)


channel_heads 0.1.0 | root /Users/guypi/Projects/channel-heads


### 1 · The transfer-learning design: 5 dimensionless features

Mars valley networks are larger and lower-resolution than Earth basins, so the model
describes a confluence with **5 scale-free geometric features**. Scale-free geometry
is what lets an Earth-trained model apply to Mars without rescaling.

In [2]:
feat_desc = {
 'orientation_diff_deg':  'angle between the two branches approaching the confluence',
 'headhead_dist_norm':    'head-to-head distance, normalized by branch length',
 'apex_angle_deg':        'interior apex angle at the confluence',
 'strahler_order_diff':   'difference in Strahler order of the two branches',
 'proximity_profile_norm':'normalized closest-approach profile of the two basins'}
pd.DataFrame({'feature': MODEL_FEATURES,
              'meaning': [feat_desc[f] for f in MODEL_FEATURES]})

,feature,meaning
0,orientation_diff_deg,angle between the two branches approaching the...
1,headhead_dist_norm,"head-to-head distance, normalized by branch le..."
2,apex_angle_deg,interior apex angle at the confluence
3,strahler_order_diff,difference in Strahler order of the two branches
4,proximity_profile_norm,normalized closest-approach profile of the two...


### 2 · The frozen production artifacts (never overwrite, never reorder)

The production classifier and CNN are preserved as-is. We load the production
XGBoost and confirm its **feature order** matches the contract above and its
**decision threshold** is the frozen 0.577406.

In [3]:
from xgboost import XGBClassifier
prod = MODELS / 'xgb_touching_classifier.json'
if prod.exists():
    m = XGBClassifier(); m.load_model(str(prod))
    names = m.get_booster().feature_names
    print('production model feature order:', names)
    print('matches frozen 5-feature contract:', names == MODEL_FEATURES)
thr = MODELS / 'optimal_threshold_geom_only.txt'
print('production threshold (frozen):', 0.577406)
print('5-class patch + 4-D CNN embedding are the other frozen contracts')

production model feature order: ['orientation_diff_deg', 'headhead_dist_norm', 'apex_angle_deg', 'strahler_order_diff', 'proximity_profile_norm']
matches frozen 5-feature contract: True
production threshold (frozen): 0.577406
5-class patch + 4-D CNN embedding are the other frozen contracts


### 3 · The three regimes = calibration uncertainty

The pipeline is run at three network *complexities* (regA/B/C). No single pruning is
"correct"; reporting the spread of Mars results across all three **is** the
calibration-uncertainty estimate. They are frozen upstream of every trained model.

In [4]:
from channel_heads.regimes import REGIMES
pd.DataFrame([{'regime': r.name, 'threshold_km2': r.threshold_km2,
               'pre_remove_max_order': r.pre_remove_max_order,
               'order_gap_to_prune': r.order_gap_to_prune,
               'character': c} for r, c in zip(REGIMES.values(),
              ['densest base / aggressive tip removal',
               'sparsest base / minimal pruning', 'intermediate density'])])

,regime,threshold_km2,pre_remove_max_order,order_gap_to_prune,character
0,regA,0.05,2,4,densest base / aggressive tip removal
1,regB,0.25,1,4,sparsest base / minimal pruning
2,regC,0.10,1,4,intermediate density


### 4 · The unit contract

A distance measured in pixels/arc-degrees is meaningless until converted with the
basin's latitude — and Mars uses a different planetary radius. All conversions go
through `channel_heads.units`, the single source of truth that keeps Earth and Mars
numerically comparable.

In [5]:
from channel_heads.units import compute_meters_per_degree
pd.DataFrame({'latitude_deg': [10, 36, 60],
              'm_per_deg_lon': [round(compute_meters_per_degree(l), 1) for l in (10, 36, 60)]})

,latitude_deg,m_per_deg_lon
0,10,110083.5
1,36,99775.8
2,60,78438.9


**Takeaway.** The pipeline is a chain of stages (0→14) that turn raw topography
into a calibrated, regime-bracketed estimate of martian channel-head coupling. The
contracts above (5 features, 5-class patch, frozen production models, the regimes,
`units.py`) are invariant; the rest of these notebooks build on them stage by stage.